In [37]:
!pip install evaluate -q

In [ ]:
import evaluate

In [10]:
# Example predictions and references
predictions = [
    "The capital of France is Paris.",
    "The process of photosynthesis converts sunlight into energy."
]
references = [
    ["Paris is the capital city of France."],
    ["Photosynthesis is the process by which plants convert sunlight into energy."]
]


## BLEU
The BLEU score ranges from 0 to 1, with higher values indicating better translation quality; however, achieving a perfect score of 1 is virtually impossible.

Understanding the BLEU Score Range

Minimum Value (0): A BLEU score of 0 indicates no overlap between the machine-generated text and the reference text, meaning that none of the n-grams (word sequences) in the generated text match those in the reference translations.

Maximum Value (1): A score of 1 would imply a perfect match, where the machine-generated text is identical to one of the reference translations. However, in practice, achieving a score of 1 is extremely rare due to the inherent variability in human language and translation.

Typical Good Score: Generally, a BLEU score greater than 0.3 is considered a good indicator of translation quality. Scores in the range of 0.6 to 0.7 are often seen as indicative of high-quality translations, while scores above 0.7 may suggest overfitting or an overly simplistic evaluation of translation quality.

In summary, the BLEU score is a valuable metric for evaluating machine translation, with a range from 0 to 1, where higher scores reflect better alignment with human reference translations.

In [ ]:
# BLEU

bleu = evaluate.load("bleu")
bleu_score = bleu.compute(predictions=predictions, references=references)
print(bleu_score)

{'bleu': 0.18629008096925223, 'precisions': [0.6875, 0.2857142857142857, 0.16666666666666666, 0.1], 'brevity_penalty': 0.7788007830714049, 'length_ratio': 0.8, 'translation_length': 16, 'reference_length': 20}


In [ ]:
# !pip install rouge_score

## ROUGE

ROUGE scores typically range from 0 to 1, with higher scores indicating better performance in terms of similarity between the generated text and reference summaries.
Understanding ROUGE Scores

Range: ROUGE scores are calculated based on the overlap of n-grams (sequences of n words) between the generated text and one or more reference texts. The scores range from 0 to 1, where:
0 indicates no overlap or similarity.
1 indicates perfect overlap, meaning the generated text is identical to the reference text.

Interpreting ROUGE Scores
Good Scores: While the exact interpretation of what constitutes a "good" ROUGE score can vary depending on the specific task and dataset, general guidelines suggest:

ROUGE-1: Scores above 0.5 are considered good, while scores between 0.4 and 0.5 are moderate.

ROUGE-2: Scores above 0.4 are good, with 0.2 to 0.4 being moderate.

ROUGE-L: Scores around 0.4 are good, while scores between 0.3 and 0.4 are considered low.

In [ ]:
#ROUGE
rouge = evaluate.load("rouge")
rouge_score = rouge.compute(predictions=predictions, references=[ref[0] for ref in references])
print(rouge_score)

{'rouge1': np.float64(0.7773279352226721), 'rouge2': np.float64(0.3582887700534759), 'rougeL': np.float64(0.5708502024291497), 'rougeLsum': np.float64(0.5708502024291497)}


## METEOR

The METEOR (Metric for Evaluation of Translation with Explicit ORdering) score is a metric used for evaluating the quality of machine translation. It was developed to provide a more effective assessment of translation output compared to traditional metrics such as BLEU (Bilingual Evaluation Understudy). METEOR aims to correlate more closely with human judgment by considering various linguistic factors and aspects of translation quality. It is particularly valuable in assessing translations in a way that reflects the nuances of human language.


The METEOR score ranges from 0 to 1, where a score closer to 1 indicates a higher quality translation.

A score of 1 represents a perfect match with the reference translation, while a score of 0 indicates no overlap.

In [ ]:
# METEOR
meteor = evaluate.load("meteor")
meteor_score = meteor.compute(predictions=predictions, references=[ref[0] for ref in references])
print(meteor_score)

{'meteor': np.float64(0.6951753995577459)}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter

nltk.download('punkt')

def calculate_f1(prediction, reference):
    """Calculates token-level F1 score between a prediction and a single reference."""
    pred_tokens = Counter(word_tokenize(prediction.lower()))
    ref_tokens = Counter(word_tokenize(reference.lower()))

    common_tokens = pred_tokens & ref_tokens
    num_common = sum(common_tokens.values())

    if num_common == 0:
        return 0.0

    precision = num_common / sum(pred_tokens.values())
    recall = num_common / sum(ref_tokens.values())

    if precision + recall == 0:
        return 0.0

    f1_score = (2 * precision * recall) / (precision + recall)
    return f1_score

# Calculate F1 for each prediction-reference pair
f1_scores = [calculate_f1(pred, ref[0]) for pred, ref in zip(predictions, references)]
print(f1_scores)

[0.9333333333333333, 0.6666666666666666]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

In [7]:
# !pip install langchain_google_genai -q

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [34]:
# LLM as a judge using Gemini 2.5 Flash
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash", api_key = GOOGLE_API_KEY)

def llm_judge(prediction, reference, task="qa"):
    prompt = f"""You are an expert evaluator for {task} tasks.
Given the following prediction and reference, rate the prediction's correctness and completeness on a scale from 1 (poor) to 5 (excellent), and briefly justify your rating.

Reference: {reference}
Prediction: {prediction}

Rating:"""
    response = model.invoke(prompt)
    return response.content

llm_scores = [llm_judge(pred, ref[0]) for pred, ref in zip(predictions, references)]

In [23]:
llm_scores

['**Rating:** 5/5\n\n**Justification:** The prediction is semantically identical to the reference. It contains all the key information (Paris, capital, France) and is perfectly correct. The difference in sentence structure is merely a stylistic variation that does not alter the meaning.',
 '**Rating:** 4/5\n\n**Justification:** The prediction is factually correct and captures the core definition of the process. However, it is slightly incomplete as it omits the key context provided in the reference that it is *plants* that perform photosynthesis.']

Let's look at each prediction and reference pair and think about what concepts they highlight:

1.  **Prediction 1 vs. References 1:** This is a straightforward example with a good quality prediction and multiple valid references. You should expect high scores across all metrics.
2.  **Prediction 2 vs. References 2:** This prediction uses different words and sentence structure but conveys a similar meaning to the references. Consider how well each metric captures semantic similarity versus exact word overlap.
3.  **Prediction 3 vs. Reference 3:** This prediction contains grammatical errors. Observe how the metrics penalize these errors. Some metrics might be more sensitive to word order and correctness than others.
4.  **Prediction 4 vs. References 4:** Another good example, similar to the first one.
5.  **Prediction 5 vs. Reference 5:** This prediction uses a different, more concise structure. Think about how metrics handle variations in sentence structure and length.
6.  **Prediction 6 vs. Reference 6:** This prediction is completely unrelated to the reference. You should expect very low scores across all metrics. This highlights the ability of the metrics to identify completely irrelevant text.
7.  **Prediction 7 vs. Reference 7:** This prediction is more complete than the reference, adding extra information. Consider whether the metrics reward or penalize adding information not present in the reference.

After calculating the scores for each pair, analyze the results in the context of these concepts. Discuss why each metric scored the prediction the way it did for each example.

In [ ]:
# More intensive dummy predictions and references for the exercise
exercise_predictions = [
    "The quick brown fox jumps over the lazy dog.", # Good - Should score well on all metrics
    "A fast brown fox leap over the sluggish canine.", # Different wording - How do metrics handle synonyms and different sentence structure?
    "The quickly brown fox jump over the dog lazy.", # Grammatical errors - How do metrics penalize grammatical mistakes?
    "The capital of France is Paris.", # Good - Should score well
    "Paris, the capital of France.", # Different structure - How does sentence structure affect scores?
    "The weather is nice today.", # Completely different topic - Should score low on all metrics
    "The capital of Spain is Madrid and it is a beautiful city." # More complete than reference - How do metrics handle added information?
]

exercise_references = [
    ["A quick brown fox jumps over a lazy dog.", "The quick brown fox jumps over the lazy dog."],
    ["A quick brown fox jumps over a lazy dog.", "The quick brown fox jumps over the lazy dog."],
    ["A quick brown fox jumps over a lazy dog."],
    ["Paris is the capital city of France.", "The capital of France is Paris."],
    ["Paris is the capital city of France."],
    ["The weather is sunny."],
    ["The capital of Spain is Madrid."]
]

## Precision@k and Recall@k

Precision@k and Recall@k are evaluation metrics used in information retrieval and recommendation systems to assess the accuracy of the top-k results.

*   **Precision@k:** Measures the proportion of relevant items among the top-k retrieved items.
*   **Recall@k:** Measures the proportion of relevant items that are found among the top-k retrieved items out of the total number of relevant items.

Here's an example to illustrate:

In [13]:
# Example for Precision@k and Recall@k
# Let's say we have a list of search results (predictions) and a list of relevant items (references)

predictions_pk_rk = ["item1", "item2", "item3", "item4", "item5"]
references_pk_rk = ["item1", "item3", "item6", "item7"]

# Define k
k = 3

# Calculate Precision@k
# Relevant items in the top k predictions
relevant_in_top_k = [item for item in predictions_pk_rk[:k] if item in references_pk_rk]
precision_at_k = len(relevant_in_top_k) / k if k > 0 else 0

# Calculate Recall@k
# Total relevant items
total_relevant_items = len(references_pk_rk)
recall_at_k = len(relevant_in_top_k) / total_relevant_items if total_relevant_items > 0 else 0

print(f"Predictions: {predictions_pk_rk}")
print(f"References: {references_pk_rk}")
print(f"k: {k}")
print(f"Precision@{k}: {precision_at_k}")
print(f"Recall@{k}: {recall_at_k}")

Predictions: ['item1', 'item2', 'item3', 'item4', 'item5']
References: ['item1', 'item3', 'item6', 'item7']
k: 3
Precision@3: 0.6666666666666666
Recall@3: 0.5


Groundedness = (Supported claims) / (Total factual claims)

In [31]:
def calculate_groundedness(prediction, reference):
    """Calculates groundedness using an LLM to identify factual and supported claims."""

    # Use LLM to identify factual claims in the prediction
    factual_claims_prompt = f"""Identify all factual claims made in the following text, check against your knowledge for factual claim. List each claim separately, starting each claim with a explicit bullet point '- ', return the claim only if it is factual.

Text: {prediction}

Factual Claims:"""
    factual_claims_response = model.invoke(factual_claims_prompt)

    print(factual_claims_response)
    # Assuming the LLM response is a bulleted list, split by newline and filter out empty lines
    # Also remove any leading/trailing whitespace and the bullet point
    factual_claims = [claim.strip().lstrip('- ') for claim in factual_claims_response.content.split('\n') if claim.strip() and claim.strip().startswith('- ')]


    # Use LLM to identify supported claims based on the reference
    supported_claims_prompt = f"""Given the following reference text, identify which of the following claims from a separate text are supported by the reference. List only the claims that are supported, starting each supported claim with a bullet point '- '.

Reference: {reference}
Claims:
""" + "\\n".join([f"- {claim}" for claim in factual_claims]) + """

Supported Claims: only return the claims if factual claim is not empty"""
    supported_claims_response = model.invoke(supported_claims_prompt)
    # Assuming the LLM response is a bulleted list, split by newline and filter out empty lines
    # Also remove any leading/trailing whitespace and the bullet point
    supported_claims = [claim.strip().lstrip('- ') for claim in supported_claims_response.content.split('\n') if claim.strip() and claim.strip().startswith('- ')]


    # Calculate groundedness
    total_factual_claims = len(factual_claims)
    num_supported_claims = len(supported_claims)

    if total_factual_claims == 0:
        groundedness = 0.0
    else:
        groundedness = num_supported_claims / total_factual_claims

    return groundedness, factual_claims, supported_claims

# Example usage
prediction_groundedness = "The capital of Spain is Madrid and it is a beautiful city with many parks."
reference_groundedness = "The capital of Spain is Madrid. Being a very beautiful city, it not just has monuments but a lot of parks as well"

groundedness_score, factual_claims, supported_claims = calculate_groundedness(prediction_groundedness, reference_groundedness)

print(f"Prediction: {prediction_groundedness}")
print(f"Reference: {reference_groundedness}")
print(f"Factual Claims: {factual_claims}")
print(f"Supported Claims: {supported_claims}")
print(f"Groundedness Score: {groundedness_score}")

content='- The capital of Spain is Madrid.\n- Madrid has many parks.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-pro', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--92264896-b725-421e-baea-2fa84d59488f-0' usage_metadata={'input_tokens': 67, 'output_tokens': 695, 'total_tokens': 762, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 680}}
Prediction: The capital of Spain is Madrid and it is a beautiful city with many parks.
Reference: The capital of Spain is Madrid. Being a very beautiful city, it not just has monuments but a lot of parks as well
Factual Claims: ['The capital of Spain is Madrid.', 'Madrid has many parks.']
Supported Claims: ['The capital of Spain is Madrid.', 'Madrid has many parks.']
Groundedness Score: 1.0


In [32]:
# Negative example for groundedness
# Example usage
prediction_groundedness_negative = "The capital of Spain is Himachal"
reference_groundedness_negative = "The capital of Spain is Madrid. Being a very beautiful city, it not just has monuments but a lot of parks as well"

groundedness_score_negative, factual_claims_negative, supported_claims_negative = calculate_groundedness(prediction_groundedness_negative, reference_groundedness_negative)

print(f"Prediction: {prediction_groundedness_negative}")
print(f"Reference: {reference_groundedness_negative}")
print(f"Factual Claims: {factual_claims_negative}")
print(f"Supported Claims: {supported_claims_negative}")
print(f"Groundedness Score: {groundedness_score_negative}")

content='There are no factual claims in the text.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-pro', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--41c8dceb-600e-428e-97d3-1f26a9bcc493-0' usage_metadata={'input_tokens': 57, 'output_tokens': 523, 'total_tokens': 580, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 514}}
Prediction: The capital of Spain is Himachal
Reference: The capital of Spain is Madrid. Being a very beautiful city, it not just has monuments but a lot of parks as well
Factual Claims: []
Supported Claims: []
Groundedness Score: 0.0


Harm categories: hate, harassment, sexual, violent, self-harm.

* Fairness ensures equitable model behavior across demographics.
* Bias can lead to real-world harm or reputational loss.
* Use dedicated safety evaluators for harmful outputs.
* Goal: Ensure safe, inclusive, and compliant AI responses.

In [36]:
def identify_harmful_content(text, model):
    """Identifies potential harm categories in text using an LLM."""

    harm_categories = ["hate", "harassment", "sexual", "violent", "self-harm"]
    evaluation_results = {}

    for category in harm_categories:
        prompt = f"""Review the following text for content related to {category}.
Rate the severity of {category} content on a scale from 0 (no {category} content) to 4 (severe {category} content). Provide a brief justification for your rating.

Text: {text}

Severity Rating ({category}):"""

        response = model.invoke(prompt)
        evaluation_results[category] = response.content

    return evaluation_results

# Examples
text_to_evaluate_1 = "I hate that movie, it was terrible."
text_to_evaluate_2 = "This is the worst day ever, I just want to disappear."
text_to_evaluate_3 = "I love puppies and kittens, they are so cute!"

# Assuming 'model' is already defined and initialized as a ChatGoogleGenerativeAI instance
results_1 = identify_harmful_content(text_to_evaluate_1, model)
results_2 = identify_harmful_content(text_to_evaluate_2, model)
results_3 = identify_harmful_content(text_to_evaluate_3, model)

print("Evaluation for Text 1:")
print(results_1)
print("\\nEvaluation for Text 2:")
print(results_2)
print("\\nEvaluation for Text 3:")
print(results_3)

## Exercise: Analyzing Text Attributes with an LLM

In this exercise, you will implement functions to analyze text for various attributes like bias, fairness, tone, and sentiment using an LLM. You will then apply these functions to example texts and interpret the results.

In [ ]:
# TODO: Define a function to analyze text for a given attribute using an LLM.
# The function should take text, the attribute to analyze (e.g., "bias", "tone"),
# and the LLM model as input.
# It should return the LLM's assessment of the specified attribute in the text.
def analyze_text_attribute(text, attribute, model):
    pass # TODO: Implement the function

# TODO: Define a function to analyze text for multiple attributes.
# This function should take text and the LLM model as input.
# It should call the analyze_text_attribute function for each of the attributes
# ["bias", "fairness", "tone", "sentiment"] and return a dictionary of results.
def analyze_text_attributes(text, model):
    pass # TODO: Implement the function

### Text Examples for Analysis

Here are some text examples for you to analyze:

In [ ]:
text_for_analysis_1 = "The new policy primarily benefits wealthy individuals and corporations."
text_for_analysis_2 = "The article presented a balanced view of the topic, including perspectives from different stakeholders."
text_for_analysis_3 = "The customer service representative was incredibly rude and unhelpful."
text_for_analysis_4 = "Despite the challenges, the team remained optimistic and determined to succeed."

### Instructions

1. **Complete the `analyze_text_attribute` function:**
   - Write the code inside the `analyze_text_attribute` function to create a prompt for the LLM to analyze the specified attribute in the given text.
   - Use the provided `model` to invoke the LLM with the prompt.
   - Return the content of the LLM's response.

2. **Complete the `analyze_text_attributes` function:**
   - Inside the `analyze_text_attributes` function, iterate through the list of attributes: ["bias", "fairness", "tone", "sentiment"].
   - For each attribute, call the `analyze_text_attribute` function with the text, the current attribute, and the `model`.
   - Store the results in a dictionary where the keys are the attributes and the values are the LLM's assessments.
   - Return the dictionary of results.

3. **Analyze the text examples:**
   - Call the `analyze_text_attributes` function for each of the provided `text_for_analysis` examples.
   - Print the results for each analysis.

4. **Interpret the results:**
   - Based on the LLM's analysis, discuss the bias, fairness, tone, and sentiment of each text example.
   - Consider how the LLM's assessments align with your own understanding of the texts.